# 2. OpenVLA model architecture

Notebook OpenVLA (Open Vision-Language-Action), image and instruction/droneaction. 

**target: **
- OpenVLA encoder + LLM 
- load and parameter
- Action Tokenizer 
- Prompt format and inferenceprocess

## 2.1 environment

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# path
PROJECT_ROOT = "/root/autodl-tmp/claude/UAV-Flow/OpenVLA-UAV"
sys.path.insert(0, PROJECT_ROOT)

MODEL_PATH = "/root/autodl-tmp/claude/models/models--openvla--openvla-7b/snapshots/47a0ec7fc4ec123775a391911046cf33cf9ed83f"

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f": {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2.2 OpenVLA 

OpenVLA is **7B parameter** --action, as follows: 

```
input: drone FPV image (224x224) + instruction
↓
┌──────────────────────────────────────────────────────┐
│ encoder 1: SigLIP-ViT () │
│ encoder 2: DINOv2-ViT ( between) │
│ ↓ + MLP Projector │
│ language model: Llama 2 7B │
│ ↓ │
│ output: N action token → control │
└──────────────────────────────────────────────────────┘
```

: **action token**, exploitationpre-training LLM large can action. 

## 2.3 load

In [ ]:
# registerdefines HuggingFace AutoClass
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticProcessor, PrismaticImageProcessor
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor, AutoModelForVision2Seq

AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

print("OpenVLA registercomplete!")

In [ ]:
# load Processor (containsimageprocess + text tokenizer)
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

print(f"Processor type: {type(processor).__name__}")
print(f"Tokenizer large small: {processor.tokenizer.vocab_size}")
print(f" large long: {processor.tokenizer.model_max_length}")

In [ ]:
# load (bfloat16 precision, 14GB)
print("load OpenVLA-7B (need to 1)...")
model = AutoModelForVision2Seq.from_pretrained(
MODEL_PATH,
torch_dtype=torch.bfloat16,
low_cpu_mem_usage=True,
trust_remote_code=True,
).to("cuda:0")
model.eval()

print(f"\nloadcomplete!")
print(f"type: {type(model).__name__}")
print(f" use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2.4 analyze

In [ ]:
# 
print("OpenVLA:")
print("=" * 50)
for name, module in model.named_children():
n_params = sum(p.numel() for p in module.parameters()) / 1e6
print(f" {name:30s} | {n_params:>8.1f}M params")

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nparameter: {total_params:.2f}B")

In [ ]:
# encoder
print("encoder (Vision Backbone):")
print("=" * 50)
vb = model.vision_backbone
print(f" Featurizer (SigLIP): {sum(p.numel() for p in vb.featurizer.parameters()) / 1e6:.1f}M params")
print(f" Fused Featurizer (DINOv2): {sum(p.numel() for p in vb.fused_featurizer.parameters()) / 1e6:.1f}M params")

# Projector
print(f"\nProjector (MLP): {sum(p.numel() for p in model.projector.parameters()) / 1e6:.1f}M params")

# LLM Backbone
print(f"\nLLM Backbone (Llama 2): {sum(p.numel() for p in model.language_model.parameters()) / 1e6:.1f}M params")

In [ ]:
# configure
config = model.config
print("configure:")
print(f" image: {config.image_sizes}")
print(f" action bin: {config.n_action_bins}")
print(f" Patch (SigLIP): {vb.featurizer.patch_embed.num_patches}")

## 2.5 Action Tokenizer: action ↔ Token

OpenVLA new: **actionvaluemap token**, from use LLM can. 

### principle
1. `[-1, 1]` between **256 bin**
2. each bin for in token ( use token) 
3. training: action → bin index → token ID
4. inference: token ID → bin center → action

In [ ]:
from prismatic.vla.action_tokenizer import ActionTokenizer

action_tokenizer = ActionTokenizer(processor.tokenizer)

print(f"Bin: {action_tokenizer.n_bins}")
print(f"action: [{action_tokenizer.min_action}, {action_tokenizer.max_action}]")
print(f"action token ID: {action_tokenizer.action_token_begin_idx}")
print(f" large small: {processor.tokenizer.vocab_size}")
print(f"\n before 5 bin edge: {action_tokenizer.bins[:5]}")
print(f" before 5 bin in: {action_tokenizer.bin_centers[:5]}")

In [ ]:
#: action → token string
example_actions = np.array([
[0.1, -0.05, 0.0, 0.2], # Move forward + + 
[-0.3, 0.1, 0.05, -0.15], # Move backward + + above + 
[0.0, 0.0, 0.0, 0.0], # Hover (action)
])

print("action:")
print("=" * 60)
for i, action in enumerate(example_actions):
token_str = action_tokenizer(action)
print(f"\naction {i}: {action}")
print(f"Token: {token_str}")

In [ ]:
#: token ID → action ( toward)
print(":")
print("=" * 60)

# simulation: from bin index value
for i, action in enumerate(example_actions):
# 
clipped = np.clip(action, -1, 1)
bin_indices = np.digitize(clipped, action_tokenizer.bins)
# 
recovered = action_tokenizer.bin_centers[np.clip(bin_indices - 1, 0, 254)]
error = np.abs(action - recovered)
print(f"\n: {action}")
print(f": {recovered}")
print(f"error: {error} ( large {error.max():.4f})")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# graph: bin 
bins = action_tokenizer.bins
centers = action_tokenizer.bin_centers
axes[0].bar(range(len(centers)), centers, width=1.0, color='steelblue', alpha=0.6)
axes[0].set_xlabel('Bin Index')
axes[0].set_ylabel('Bin Center Value')
axes[0].set_title(f'256 Bins: [-1, 1] Uniform Discretization')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

# graph: precision
test_values = np.linspace(-1, 1, 1000)
bin_indices = np.digitize(test_values, bins)
recovered_values = centers[np.clip(bin_indices - 1, 0, 254)]
errors = np.abs(test_values - recovered_values)

axes[1].plot(test_values, errors * 1000, color='coral')
axes[1].set_xlabel('Original Value')
axes[1].set_ylabel('Discretization Error (x1000)')
axes[1].set_title(f'Quantization Error (max={errors.max():.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n each bin: {2/256:.6f}")
print(f" large error: {errors.max():.6f}")
print(f"for UAV action ( ~0.1m), error {errors.max() * 0.5:.4f}m -> ")

## 2.6 Prompt format

OpenVLA use Prompt formattaskinstructionconvertinput: 

In [ ]:
from prismatic.models.backbones.llm.prompting import PurePromptBuilder

# OpenVLA Prompt format
prompt_builder = PurePromptBuilder("openvla")

# simulation training prompt build
proprio = "0.1,0.2,-0.1,15.3" # before state: x,y,z,yaw
instruction = "fly toward the tree on the left side"
action_tokens = action_tokenizer(np.array([0.1, -0.05, 0.0, 0.2])) # simulationaction

prompt_builder.add_turn("human", f"Current State: {proprio}, What action should the uav take to {instruction}?")
prompt_builder.add_turn("gpt", action_tokens)

full_prompt = prompt_builder.get_prompt()
print(" Prompt:")
print("=" * 60)
print(full_prompt)
print("=" * 60)

# inference Prompt ( not GPT reply)
print("\ninference use Prompt:")
inference_prompt = f"In: What action should the robot take to {instruction}?\nOut:"
print(inference_prompt)

In [ ]:
# Token 
tokens = processor.tokenizer(full_prompt, add_special_tokens=True)
input_ids = tokens.input_ids

print(f"Token: {len(input_ids)}")
print(f"\n before 10 token ID: {input_ids[:10]}")
print(f"decoding: {processor.tokenizer.decode(input_ids[:10])}")
print(f"\n after 10 token ID: {input_ids[-10:]}")
print(f"decoding: {processor.tokenizer.decode(input_ids[-10:])}")

# action token
action_token_start = action_tokenizer.action_token_begin_idx
action_ids = [t for t in input_ids if t >= action_token_start]
print(f"\naction token: {len(action_ids)}")
print(f"action token IDs: {action_ids}")

## 2.7 imageprocessprocess

In [ ]:
from PIL import Image

# loadtestimage
DATA_DIR = "/root/autodl-tmp/claude/data/uav_flow_subset"
sample_dir = sorted(os.listdir(DATA_DIR))[0]
img_path = os.path.join(DATA_DIR, sample_dir, "000005.jpg")
image = Image.open(img_path).convert("RGB")

print(f"image: {image.size}")

# through Processor process
instruction = "fly toward the tree"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"
inputs = processor(prompt, image)

print(f"\nProcessor output:")
for key, val in inputs.items():
if hasattr(val, 'shape'):
print(f" {key}: shape={val.shape}, dtype={val.dtype}")
else:
print(f" {key}: {type(val).__name__}, len={len(val)}")

In [ ]:
# Visualization: image vs input
pixel_values = inputs['pixel_values']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# image
axes[0].imshow(image)
axes[0].set_title(f'Original Image\n{image.size}')
axes[0].axis('off')

# SigLIP input ( before 3)
siglip_img = pixel_values[0,:3].permute(1, 2, 0).numpy()
siglip_img = (siglip_img - siglip_img.min()) / (siglip_img.max() - siglip_img.min())
axes[1].imshow(siglip_img)
axes[1].set_title(f'SigLIP Input\n224x224 (semantic)')
axes[1].axis('off')

# DINOv2 input ( after 3)
dino_img = pixel_values[0, 3:6].permute(1, 2, 0).numpy()
dino_img = (dino_img - dino_img.min()) / (dino_img.max() - dino_img.min())
axes[2].imshow(dino_img)
axes[2].set_title(f'DINOv2 Input\n224x224 (spatial)')
axes[2].axis('off')

plt.suptitle(f'Dual Vision Encoder Inputs\npixel_values shape: {pixel_values.shape}', fontsize=13)
plt.tight_layout()
plt.show()

## 2.8 inference

In [ ]:
# use timesinference ( UAV fine-tuning, thereforeresult not will has)
instruction = "fly toward the tree on the left side"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)

print(f"instruction: {instruction}")
print(f"image: {img_path}")
print(f"\ngenerateaction token...")

with torch.no_grad():
# generate token ( not use predict_action, because has UAV norm_stats)
generated_ids = model.generate(
**inputs,
max_new_tokens=4, # 4 action
do_sample=False,
)

# generate token
new_tokens = generated_ids[0, -4:].cpu().numpy()
print(f"generate token IDs: {new_tokens}")

# 
decoded_actions = action_tokenizer.decode_token_ids_to_actions(new_tokens)
print(f"normalizationaction [-1,1]: {decoded_actions}")

dim_names = ['x ( before after)', 'y ()', 'z ( above below)', 'yaw ()']
print(f"\naction:")
for name, val in zip(dim_names, decoded_actions):
print(f" {name}: {val:+.4f}")

print("\n[note] for UAV fine-tuning, resultprocess. ")
print("fine-tuning after in Notebook 03 and 04 in use. ")

## 2.9 datasummary

```
training:
Image ──→ Processor.image_processor ──→ pixel_values [6, 224, 224]
Text ──→ Processor.tokenizer ──→ input_ids [N]
Action──→ ActionTokenizer ──→ action tokens ( labels)

Model forward: pixel_values + input_ids → logits → CrossEntropyLoss

inference:
Image + Prompt ──→ Processor ──→ pixel_values + input_ids
Model.generate ──→ action token IDs
ActionTokenizer.decode ──→ normalizationaction [-1, 1]
De-normalize ( use q01/q99) ──→ actionvalue
```

In [ ]:
# 
del model
torch.cuda.empty_cache()
print(f"complete. use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## small 

- OpenVLA use **encoder** (SigLIP + DINOv2) image
- through **MLP Projector** between 
- exploitation **Llama 2 7B** can action token
- **Action Tokenizer** action 256 bin, precision (error < 0.004)
- Prompt format: `In: What action should the robot take to {instruction}?\nOut:`

** below:** in Notebook 03 in, use LoRA for OpenVLA droneactionfine-tuning. 